In [1]:
df = spark.sql("SELECT * FROM cor_project.silver.swell_metrics where coast_name = 'Matamoros'")

In [2]:
df.show(1)

+-------+----------+-------------------+----+-------------+------------------+------------------+------------------+-------------+------------------+------------------+------------------+-------------+-----------+--------------+--------------------+--------------------+------------------------+
|     id|coast_name|           datetime|year|wind_speed_ms|wind_direction_deg|wind_cos_direction|wind_sin_direction|wave_height_m|wave_direction_deg|wave_cos_direction|wave_sin_direction|wave_period_s|wave_energy|wave_steepness| ingestion_timestamp|         source_file|transformation_timestamp|
+-------+----------+-------------------+----+-------------+------------------+------------------+------------------+-------------+------------------+------------------+------------------+-------------+-----------+--------------+--------------------+--------------------+------------------------+
|1350058| Matamoros|1990-01-01 00:00:00|1990|        12.28|            192.99|           -0.9744|           -0.2

In [3]:
variables = ['wind_speed_ms', 'wave_height_m', 'wave_period_s', 'wave_energy']
threshold_95 = {}
threshold_97 = {}
threshold_99 = {}
for var in variables:
    quantiles = df.approxQuantile(var, [0.95, 0.97, 0.99], 0.01)
    threshold_95[var] = quantiles[0]
    threshold_97[var] = quantiles[1]
    threshold_99[var] = quantiles[2]

In [8]:
# generar un df con id, coast_name, year, month, wind_speed_ms_over_95, wind_speed_ms_over_97, wind_speed_ms_over_99, wave_height_m_over_95, wave_height_m_over_97, wave_height_m_over_99, wave_period_s_over_95, wave_period_s_over_97, wave_period_s_over_99, wave_energy_over_95, wave_energy_over_97, wave_energy_over_99
import pyspark.sql.functions as F

result_df = df.select(
    "id",
    "coast_name",
    F.year("datetime").alias("year"),
    F.month("datetime").alias("month"),
    (F.col("wind_speed_ms") > threshold_95["wind_speed_ms"]).alias("wind_speed_ms_over_95"),
    (F.col("wind_speed_ms") > threshold_97["wind_speed_ms"]).alias("wind_speed_ms_over_97"),
    (F.col("wind_speed_ms") > threshold_99["wind_speed_ms"]).alias("wind_speed_ms_over_99"),

    (F.col("wave_height_m") > threshold_95["wave_height_m"]).alias("wave_height_m_over_95"),
    (F.col("wave_height_m") > threshold_97["wave_height_m"]).alias("wave_height_m_over_97"),
    (F.col("wave_height_m") > threshold_99["wave_height_m"]).alias("wave_height_m_over_99"),

    (F.col("wave_period_s") > threshold_95["wave_period_s"]).alias("wave_period_s_over_95"),
    (F.col("wave_period_s") > threshold_97["wave_period_s"]).alias("wave_period_s_over_97"),
    (F.col("wave_period_s") > threshold_99["wave_period_s"]).alias("wave_period_s_over_99"),

    (F.col("wave_energy") > threshold_95["wave_energy"]).alias("wave_energy_over_95"),
    (F.col("wave_energy") > threshold_97["wave_energy"]).alias("wave_energy_over_97"),
    (F.col("wave_energy") > threshold_99["wave_energy"]).alias("wave_energy_over_99")
)

In [9]:
# contar el número de filas donde cada una de las columnas de over_95, over_97 y over_99 es True agrupando por año y mes
result_df.groupBy("year", "month").agg(
    F.sum("wind_speed_ms_over_95").alias("wind_speed_ms_over_95_count"),
    F.sum("wind_speed_ms_over_97").alias("wind_speed_ms_over_97_count"),
    F.sum("wind_speed_ms_over_99").alias("wind_speed_ms_over_99_count"),

    F.sum("wave_height_m_over_95").alias("wave_height_m_over_95_count"),
    F.sum("wave_height_m_over_97").alias("wave_height_m_over_97_count"),
    F.sum("wave_height_m_over_99").alias("wave_height_m_over_99_count"),

    F.sum("wave_period_s_over_95").alias("wave_period_s_over_95_count"),
    F.sum("wave_period_s_over_97").alias("wave_period_s_over_97_count"),
    F.sum("wave_period_s_over_99").alias("wave_period_s_over_99_count"),

    F.sum("wave_energy_over_95").alias("wave_energy_over_95_count"),
    F.sum("wave_energy_over_97").alias("wave_energy_over_97_count"),
    F.sum("wave_energy_over_99").alias("wave_energy_over_99_count")
).show()

AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "sum(wind_speed_ms_over_95)" due to data type mismatch: The first parameter requires the "NUMERIC" or "ANSI INTERVAL" type, however "wind_speed_ms_over_95" has the type "BOOLEAN". SQLSTATE: 42K09;
'Aggregate [year#11580, month#11581], [year#11580, month#11581, sum(wind_speed_ms_over_95#11582) AS wind_speed_ms_over_95_count#11594, sum(wind_speed_ms_over_97#11583) AS wind_speed_ms_over_97_count#11595, sum(wind_speed_ms_over_99#11584) AS wind_speed_ms_over_99_count#11596, sum(wave_height_m_over_95#11585) AS wave_height_m_over_95_count#11597, sum(wave_height_m_over_97#11586) AS wave_height_m_over_97_count#11598, sum(wave_height_m_over_99#11587) AS wave_height_m_over_99_count#11599, sum(wave_period_s_over_95#11588) AS wave_period_s_over_95_count#11600, sum(wave_period_s_over_97#11589) AS wave_period_s_over_97_count#11601, sum(wave_period_s_over_99#11590) AS wave_period_s_over_99_count#11602, sum(wave_energy_over_95#